In [ ]:
!pip install pyiqa --no-deps

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.fft
from torchvision import models
import glob
import os
import random
from pathlib import Path
from PIL import Image
import torch
from torch.utils.data import Dataset
import torchvision.transforms.functional as TF
from torchvision import transforms
import matplotlib.pyplot as plt
import pyiqa
from torch.utils.data import DataLoader
from tqdm import tqdm
import numpy as np

from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/Checkpoints/KASIRNet
!mkdir -p /content/drive/MyDrive/Results/KASIRNet
!mkdir -p /content/drive/MyDrive/Visualizations/KASIRNet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 89.8 MB/s eta 0:00:00
Mounted at /content/drive


In [5]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("rajneesh231/salt-and-pepper-noise-images")
print("Path to dataset files:", path)
path = kagglehub.dataset_download("kmljts/reside-6k")
print("Path to dataset files:", path)
path = kagglehub.dataset_download("eshapahwa1302/kohlerdeblur")
print("Path to dataset files:", path)
path = kagglehub.dataset_download("shivakanthsujit/jrdr-deraining-dataset")
print("Path to dataset files:", path)
path = kagglehub.dataset_download("adeshpal/image-deraining")
print("Path to dataset files:", path)
path = kagglehub.dataset_download("chenchienying/deraindataset")
print("Path to dataset files:", path)
path = kagglehub.dataset_download("jishnuparayilshibu/a-curated-list-of-image-deblurring-datasets")
print("Path to dataset files:", path)
path = kagglehub.dataset_download("tenxengineers/denoising-dataset-multiple-iso-levels")
print("Path to dataset files:", path)
path = kagglehub.dataset_download("mohamedkhaled201841/image-denoising")
print("Path to dataset files:", path)

Using Colab cache for faster access to the 'salt-and-pepper-noise-images' dataset.
Path to dataset files: /kaggle/input/salt-and-pepper-noise-images
Using Colab cache for faster access to the 'reside-6k' dataset.
Path to dataset files: /kaggle/input/reside-6k
Using Colab cache for faster access to the 'kohlerdeblur' dataset.
Path to dataset files: /kaggle/input/kohlerdeblur
Using Colab cache for faster access to the 'jrdr-deraining-dataset' dataset.
Path to dataset files: /kaggle/input/jrdr-deraining-dataset
Using Colab cache for faster access to the 'image-deraining' dataset.
Path to dataset files: /kaggle/input/image-deraining
Using Colab cache for faster access to the 'deraindataset' dataset.
Path to dataset files: /kaggle/input/deraindataset
Using Colab cache for faster access to the 'a-curated-list-of-image-deblurring-datasets' dataset.
Path to dataset files: /kaggle/input/a-curated-list-of-image-deblurring-datasets
Using Colab cache for faster access to the 'denoising-dataset-mul

In [ ]:
!ls '/kaggle/input/a-curated-list-of-image-deblurring-datasets/DBlur'

In [ ]:
# Step 1 — Install gdown
!pip install gdown -q

# Step 2 — Download
!gdown "1tfeBnjZX1wIhIFPl6HOzzOKOyo0GdGHl" -O /content/dataset.tar.gz

# Step 3 — Check file
!ls -lh /content/dataset.tar.gz
!file /content/dataset.tar.gz

!unzip /content/dataset.tar.gz -d /content/
print("Done!")
#!ls -lh /content/

Le flux de sortie a été tronqué et ne contient que les 5000 dernières lignes.
  inflating: /content/input/winter_building_03110.jpg  
  inflating: /content/input/beautiful_smile_00738.jpg  
  inflating: /content/input/school_night_08146.jpg  
  inflating: /content/input/sidewalk winter -grayscale -gray_06236.jpg  
  inflating: /content/input/snow_intersection_00104.jpg  
  inflating: /content/input/winter_house_02885.jpg  
  inflating: /content/input/parking_lot_12107.jpg  
  inflating: /content/input/castle_00593.jpg  
  inflating: /content/input/crossing_03457.jpg  
  inflating: /content/input/sidewalk winter -grayscale -gray_02441.jpg  
  inflating: /content/input/parking_lot_15413.jpg  
  inflating: /content/input/parking_lot_13701.jpg  
  inflating: /content/input/winter intersection -snow_01381.jpg  
  inflating: /content/input/snow_animal_02068.jpg  
  inflating: /content/input/winter_new_york_00783.jpg  
  inflating: /content/input/parking_lot_07478.jpg  
  inflating: /content/

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
import os
import random
from pathlib import Path
from PIL import Image
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader
import glob
import csv
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from tqdm import tqdm

# ==========================================================================
# KASIRNet (Omni-Restoration Network)
# Full Production Implementation
# ==========================================================================

# ==========================================
# 1. Core Components (SCWAM & Mesgrid)
# ==========================================

class MesgridCoordinateEmbedder(nn.Module):
    """
    Generates a spatial meshgrid (x, y) and embeds it into feature space.
    Gives the attention mechanism explicit geometric awareness.
    """
    def __init__(self, embed_dim):
        super(MesgridCoordinateEmbedder, self).__init__()
        self.embed = nn.Sequential(
            nn.Conv2d(2, embed_dim // 4, 1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(embed_dim // 4, embed_dim, 1)
        )

    def forward(self, x):
        B, C, H, W = x.shape
        y_coords = torch.linspace(-1, 1, H, device=x.device, dtype=x.dtype).view(1, 1, H, 1).expand(B, 1, H, W)
        x_coords = torch.linspace(-1, 1, W, device=x.device, dtype=x.dtype).view(1, 1, 1, W).expand(B, 1, H, W)
        coords = torch.cat([x_coords, y_coords], dim=1)
        return self.embed(coords)


class WindowAttention(nn.Module):
    def __init__(self, dim, window_size, num_heads):
        super(WindowAttention, self).__init__()
        self.dim = dim
        self.window_size = window_size
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = head_dim ** -0.5

        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)

    def forward(self, x):
        B_, N, C = x.shape
        qkv = self.qkv(x).reshape(B_, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)

        x = (attn @ v).transpose(1, 2).reshape(B_, N, C)
        x = self.proj(x)
        return x


class SCWAMLayer(nn.Module):
    """
    Shifted Cyclic Window Attention with Mesgrid.
    """
    def __init__(self, dim, num_heads, window_size=8, shift_size=4):
        super(SCWAMLayer, self).__init__()
        self.dim = dim
        self.num_heads = num_heads
        self.window_size = window_size
        self.shift_size = shift_size

        self.norm1 = nn.LayerNorm(dim)
        self.attn = WindowAttention(dim, window_size, num_heads)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, dim * 4),
            nn.GELU(),
            nn.Linear(dim * 4, dim)
        )
        self.grid_embedder = MesgridCoordinateEmbedder(dim)

    def forward(self, x):
        B, C, H, W = x.shape

        grid_emb = self.grid_embedder(x)
        x = x + grid_emb
        shortcut = x

        pad_l = pad_t = 0
        pad_r = (self.window_size - W % self.window_size) % self.window_size
        pad_b = (self.window_size - H % self.window_size) % self.window_size
        x = torch.nn.functional.pad(x, (pad_l, pad_r, pad_t, pad_b))
        _, _, Hp, Wp = x.shape

        x = x.flatten(2).transpose(1, 2)
        x = self.norm1(x).view(B, Hp, Wp, C)

        if self.shift_size > 0:
            shifted_x = torch.roll(x, shifts=(-self.shift_size, -self.shift_size), dims=(1, 2))
        else:
            shifted_x = x

        x_windows = self.window_partition(shifted_x, self.window_size)
        attn_windows = self.attn(x_windows)

        shifted_x = self.window_reverse(attn_windows, self.window_size, Hp, Wp)
        if self.shift_size > 0:
            x = torch.roll(shifted_x, shifts=(self.shift_size, self.shift_size), dims=(1, 2))
        else:
            x = shifted_x

        if pad_r > 0 or pad_b > 0:
            x = x[:, :H, :W, :].contiguous()

        x = x.view(B, H * W, C)

        x = shortcut.flatten(2).transpose(1, 2) + x
        x = x + self.mlp(self.norm2(x))

        return x.transpose(1, 2).view(B, C, H, W)

    def window_partition(self, x, window_size):
        B, H, W, C = x.shape
        x = x.view(B, H // window_size, window_size, W // window_size, window_size, C)
        windows = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(-1, window_size * window_size, C)
        return windows

    def window_reverse(self, windows, window_size, H, W):
        B = int(windows.shape[0] / (H * W / window_size / window_size))
        x = windows.view(B, H // window_size, W // window_size, window_size, window_size, -1)
        x = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(B, H, W, -1)
        return x


# ==========================================
# 2. Spectral Resonance Block
# ==========================================

class SpectralResonanceBlock(nn.Module):
    def __init__(self, dim):
        super(SpectralResonanceBlock, self).__init__()
        self.freq_conv = nn.Conv2d(dim * 2, dim * 2, kernel_size=1)
        self.resonance_gate = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(dim, dim // 4, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(dim // 4, dim, 1),
            nn.Sigmoid()
        )
        self.spatial_restore = nn.Sequential(
            nn.Conv2d(dim, dim, 3, 1, 1, groups=dim),
            nn.Conv2d(dim, dim, 1)
        )

    def forward(self, x):
        B, C, H, W = x.shape
        identity = x

        x_fft = torch.fft.rfft2(x, norm='backward')
        mag = torch.abs(x_fft)
        pha = torch.angle(x_fft)

        gate = self.resonance_gate(mag)
        mag_resonant = mag * (1.0 + gate)

        real_res = mag_resonant * torch.cos(pha)
        imag_res = mag_resonant * torch.sin(pha)

        f_stack = torch.cat([real_res, imag_res], dim=1)
        f_stack = self.freq_conv(f_stack)

        real_final, imag_final = torch.chunk(f_stack, 2, dim=1)
        x_fft_final = torch.complex(real_final, imag_final)

        x_restored = torch.fft.irfft2(x_fft_final, s=(H, W), norm='backward')
        return identity + self.spatial_restore(x_restored)


# ==========================================
# 3. VGG19 Semantic Feature Extractor
# ==========================================

class VGG19SemanticExtractor(nn.Module):
    def __init__(self):
        super(VGG19SemanticExtractor, self).__init__()
        vgg = models.vgg19(weights=None).features
        self.scale1 = nn.Sequential(*[vgg[i] for i in range(5)])
        self.scale2 = nn.Sequential(*[vgg[i] for i in range(5, 10)])
        self.scale3 = nn.Sequential(*[vgg[i] for i in range(10, 19)])
        self.scale4 = nn.Sequential(*[vgg[i] for i in range(19, 28)])

        for param in self.parameters():
            param.requires_grad = False

        self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer('std', torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, x):
        x = (x - self.mean) / self.std
        f1 = self.scale1(x)
        f2 = self.scale2(f1)
        f3 = self.scale3(f2)
        f4 = self.scale4(f3)
        return [f1, f2, f3, f4]


# ==========================================
# 4. Semantic Injection Module
# ==========================================

class SemanticInjection(nn.Module):
    def __init__(self, restore_dim, semantic_dim):
        super(SemanticInjection, self).__init__()
        self.semantic_proj = nn.Conv2d(semantic_dim, restore_dim, 1)
        self.channel_attn = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(restore_dim, restore_dim // 4, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(restore_dim // 4, restore_dim, 1),
            nn.Sigmoid()
        )
        self.spatial_attn = nn.Sequential(
            nn.Conv2d(restore_dim, 1, 7, 1, 3),
            nn.Sigmoid()
        )
        self.fusion = nn.Sequential(
            nn.Conv2d(restore_dim * 2 + semantic_dim, restore_dim, 1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(restore_dim, restore_dim, 3, 1, 1)
        )

    def forward(self, restore_feat, semantic_feat):
        B, C, H, W = restore_feat.shape
        identity = restore_feat

        sem_proj = self.semantic_proj(semantic_feat)
        if sem_proj.shape[2:] != restore_feat.shape[2:]:
            sem_proj = torch.nn.functional.interpolate(sem_proj, size=(H, W), mode='bilinear', align_corners=False)
            semantic_feat = torch.nn.functional.interpolate(semantic_feat, size=(H, W), mode='bilinear', align_corners=False)

        ch_attn = self.channel_attn(sem_proj)
        restore_ch = restore_feat * ch_attn

        sp_attn = self.spatial_attn(sem_proj)
        restore_sp = restore_feat * sp_attn

        fused = self.fusion(torch.cat([semantic_feat, restore_ch, restore_sp], dim=1))
        return identity + fused


# ==========================================
# 5. Enhanced Encoder/Decoder Blocks
# ==========================================

class EncoderBlock(nn.Module):
    def __init__(self, in_channels, out_channels, semantic_dim, num_heads=8, use_semantic=True):
        super(EncoderBlock, self).__init__()
        self.use_semantic = use_semantic
        self.downsample = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, 2, 1),
            nn.LeakyReLU(0.2, inplace=True)
        )
        self.srb = SpectralResonanceBlock(out_channels)
        self.scwam = SCWAMLayer(out_channels, num_heads=num_heads)
        if use_semantic:
            self.sim = SemanticInjection(out_channels, semantic_dim)

    def forward(self, x, semantic_feat=None):
        x = self.downsample(x)
        x = self.srb(x)
        if self.use_semantic and semantic_feat is not None:
            x = self.sim(x, semantic_feat)
        x = self.scwam(x)
        return x

class DecoderBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels, semantic_dim, prompt_dim=256, num_heads=8, use_semantic=True):
        super(DecoderBlock, self).__init__()
        self.use_semantic = use_semantic
        self.upsample = nn.Sequential(
            nn.ConvTranspose2d(in_channels, out_channels, 4, 2, 1),
            nn.LeakyReLU(0.2, inplace=True)
        )
        self.fusion = nn.Sequential(
            nn.Conv2d(out_channels + skip_channels, out_channels, 1),
            nn.LeakyReLU(0.2, inplace=True)
        )
        self.srb = SpectralResonanceBlock(out_channels)
        self.scwam = SCWAMLayer(out_channels, num_heads=num_heads)
        self.sft = SFTLayer(out_channels, prompt_dim)
        if use_semantic:
            self.sim = SemanticInjection(out_channels, semantic_dim)

    def forward(self, x, skip, prompt_emb, semantic_feat=None):
        x = self.upsample(x)
        if x.size(2) != skip.size(2) or x.size(3) != skip.size(3):
            x = torch.nn.functional.interpolate(x, size=(skip.size(2), skip.size(3)), mode='bilinear', align_corners=False)
        x = torch.cat([x, skip], dim=1)
        x = self.fusion(x)
        x = self.srb(x)
        if self.use_semantic and semantic_feat is not None:
            x = self.sim(x, semantic_feat)
        x = self.scwam(x)
        x = self.sft(x, prompt_emb)
        return x


class SFTLayer(nn.Module):
    def __init__(self, in_channels, prompt_dim=256):
        super(SFTLayer, self).__init__()
        self.sft_scale = nn.Conv2d(prompt_dim, in_channels, 1)
        self.sft_shift = nn.Conv2d(prompt_dim, in_channels, 1)

    def forward(self, x, prompt_emb):
        prompt = prompt_emb.view(prompt_emb.size(0), prompt_emb.size(1), 1, 1)
        prompt = prompt.expand(-1, -1, x.size(2), x.size(3))
        scale = self.sft_scale(prompt)
        shift = self.sft_shift(prompt)
        return x * (scale + 1) + shift


class GlobalContextBlock(nn.Module):
    def __init__(self, in_channels):
        super(GlobalContextBlock, self).__init__()
        self.conv_mask = nn.Conv2d(in_channels, 1, kernel_size=1)
        self.softmax = nn.Softmax(dim=2)
        self.channel_add_conv = nn.Sequential(
            nn.Conv2d(in_channels, in_channels // 2, kernel_size=1),
            nn.LayerNorm([in_channels // 2, 1, 1]),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels // 2, in_channels, kernel_size=1)
        )

    def forward(self, x):
        b, c, h, w = x.size()
        input_x = x
        input_x = input_x.view(b, c, h * w).unsqueeze(1)
        mask = self.conv_mask(x).view(b, 1, h * w)
        mask = self.softmax(mask).unsqueeze(-1)
        context = torch.matmul(input_x, mask)
        context = context.view(b, c, 1, 1)
        return x + self.channel_add_conv(context)


class PromptLearner(nn.Module):
    def __init__(self, in_channels=3, prompt_dim=256):
        super(PromptLearner, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 64, 3, 2, 1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, 3, 2, 1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 256, 3, 2, 1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.AdaptiveAvgPool2d(1)
        )
        self.fc = nn.Sequential(
            nn.Linear(256, prompt_dim),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(prompt_dim, prompt_dim)
        )

    def forward(self, x):
        feat = self.features(x)
        feat = feat.view(feat.size(0), -1)
        return self.fc(feat)


# ==========================================
# 6. KASIRNet Definition
# ==========================================

class KASIRNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=3, base_channels=64, prompt_dim=256):
        super(KASIRNet, self).__init__()
        self.vgg_extractor = VGG19SemanticExtractor()
        vgg_dims = [64, 128, 256, 512]
        ch = [base_channels, base_channels*2, base_channels*4, base_channels*6, base_channels*8]
        self.prompt_learner = PromptLearner(in_channels, prompt_dim)

        self.head_conv = nn.Sequential(
            nn.Conv2d(in_channels, ch[0], 3, 1, 1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ch[0], ch[0], 3, 1, 1)
        )
        self.head_attn = SCWAMLayer(ch[0], num_heads=2)

        self.encoder1 = EncoderBlock(ch[0], ch[1], vgg_dims[0], num_heads=4)
        self.encoder2 = EncoderBlock(ch[1], ch[2], vgg_dims[1], num_heads=4)
        self.encoder3 = EncoderBlock(ch[2], ch[3], vgg_dims[2], num_heads=8)
        self.encoder4 = EncoderBlock(ch[3], ch[4], vgg_dims[3], num_heads=8)

        self.bottleneck = nn.Sequential(
            GlobalContextBlock(ch[4]),
            SpectralResonanceBlock(ch[4]),
            SCWAMLayer(ch[4], num_heads=8)
        )

        self.decoder4 = DecoderBlock(ch[4], ch[3], ch[3], vgg_dims[3], prompt_dim, num_heads=8)
        self.decoder3 = DecoderBlock(ch[3], ch[2], ch[2], vgg_dims[2], prompt_dim, num_heads=8)
        self.decoder2 = DecoderBlock(ch[2], ch[1], ch[1], vgg_dims[1], prompt_dim, num_heads=4)
        self.decoder1 = DecoderBlock(ch[1], ch[0], ch[0], vgg_dims[0], prompt_dim, num_heads=4)

        self.tail = nn.Sequential(
            nn.Conv2d(ch[0], ch[0], 3, 1, 1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ch[0], out_channels, 3, 1, 1)
        )

    def check_image_size(self, x):
        _, _, h, w = x.size()
        mod_pad_h = (16 - h % 16) % 16
        mod_pad_w = (16 - w % 16) % 16
        x = torch.nn.functional.pad(x, (0, mod_pad_w, 0, mod_pad_h), mode='replicate')
        return x, h, w

    def forward(self, x):
        x_padded, H_orig, W_orig = self.check_image_size(x)
        with torch.no_grad():
            vgg_feats = self.vgg_extractor(x_padded)

        prompt_emb = self.prompt_learner(x_padded)

        x0 = self.head_conv(x_padded)
        x0 = self.head_attn(x0)

        x1 = self.encoder1(x0, vgg_feats[0])
        x2 = self.encoder2(x1, vgg_feats[1])
        x3 = self.encoder3(x2, vgg_feats[2])
        x4 = self.encoder4(x3, vgg_feats[3])

        x_mid = self.bottleneck(x4)

        d4 = self.decoder4(x_mid, x3, prompt_emb, vgg_feats[3])
        d3 = self.decoder3(d4, x2, prompt_emb, vgg_feats[2])
        d2 = self.decoder2(d3, x1, prompt_emb, vgg_feats[1])
        d1 = self.decoder1(d2, x0, prompt_emb, vgg_feats[0])

        out = self.tail(d1)
        out = out + x_padded
        out = out[:, :, :H_orig, :W_orig].clamp(0, 1)

        return out, prompt_emb


# ==========================================
# 8. Loss Functions
# ==========================================

class IdentityLoss(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = models.resnet50(weights=None)
        self.backbone = nn.Sequential(*list(resnet.children())[:-1])
        for param in self.backbone.parameters():
            param.requires_grad = False
        self.criterion = nn.L1Loss()

    def forward(self, pred, target):
        pred_id = self.backbone(pred).view(pred.size(0), -1)
        target_id = self.backbone(target).view(target.size(0), -1)
        return self.criterion(pred_id, target_id)


class VGGFeatureExtractor(nn.Module):
    def __init__(self, requires_grad=False):
        super().__init__()
        vgg = models.vgg19(weights=None).features
        self.slice1 = nn.Sequential(*[vgg[i] for i in range(5)])
        self.slice2 = nn.Sequential(*[vgg[i] for i in range(5, 10)])
        self.slice3 = nn.Sequential(*[vgg[i] for i in range(10, 19)])
        self.slice4 = nn.Sequential(*[vgg[i] for i in range(19, 28)])
        if not requires_grad:
            for param in self.parameters():
                param.requires_grad = False

    def forward(self, x):
        h1 = self.slice1(x)
        h2 = self.slice2(h1)
        h3 = self.slice3(h2)
        h4 = self.slice4(h3)
        return [h1, h2, h3, h4]


class OmniRestorationLoss(nn.Module):
    def __init__(self, device='cuda', fft_weight=1.0, perceptual_weight=1.0, ssim_weight=1.0, identity_weight=1.0, edge_weight=1.0):
        super().__init__()
        self.fft_weight = fft_weight
        self.perceptual_weight = perceptual_weight
        self.ssim_weight = ssim_weight
        self.identity_weight = identity_weight
        self.edge_weight = edge_weight
        self.l1 = nn.L1Loss()
        self.vgg = VGGFeatureExtractor(requires_grad=False).to(device)
        self.id_loss_fn = IdentityLoss().to(device)

        sobel_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=torch.float32)
        sobel_y = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=torch.float32)
        self.register_buffer('sobel_x', sobel_x.view(1, 1, 3, 3).repeat(3, 1, 1, 1))
        self.register_buffer('sobel_y', sobel_y.view(1, 1, 3, 3).repeat(3, 1, 1, 1))

        try:
            import pyiqa
            self.ssim_metric = pyiqa.create_metric('ssim', device=device)
            self.use_pyiqa = True
        except:
            self.use_pyiqa = False

    def compute_edge(self, x):
        edge_x = torch.nn.functional.conv2d(x, self.sobel_x, padding=1, groups=3)
        edge_y = torch.nn.functional.conv2d(x, self.sobel_y, padding=1, groups=3)
        edge = torch.sqrt(edge_x**2 + edge_y**2 + 1e-6)
        return edge

    def compute_ssim(self, pred, target, window_size=11, C1=0.01**2, C2=0.03**2):
        mu_pred = torch.nn.functional.avg_pool2d(pred, window_size, stride=1, padding=window_size//2)
        mu_target = torch.nn.functional.avg_pool2d(target, window_size, stride=1, padding=window_size//2)
        mu_pred_sq = mu_pred ** 2
        mu_target_sq = mu_target ** 2
        mu_pred_target = mu_pred * mu_target
        sigma_pred_sq = torch.nn.functional.avg_pool2d(pred**2, window_size, stride=1, padding=window_size//2) - mu_pred_sq
        sigma_target_sq = torch.nn.functional.avg_pool2d(target**2, window_size, stride=1, padding=window_size//2) - mu_target_sq
        sigma_pred_target = torch.nn.functional.avg_pool2d(pred*target, window_size, stride=1, padding=window_size//2) - mu_pred_target
        ssim = ((2*mu_pred_target + C1) * (2*sigma_pred_target + C2)) / \
               ((mu_pred_sq + mu_target_sq + C1) * (sigma_pred_sq + sigma_target_sq + C2))
        return ssim.mean()

    def forward(self, preds, targets):
        eps = 1e-6
        loss_dict = {}

        pixel_loss = self.l1(preds, targets)
        loss_dict['pixel'] = pixel_loss.item()

        pred_fft = torch.fft.rfft2(preds, norm='backward')
        target_fft = torch.fft.rfft2(targets, norm='backward')
        fft_loss = self.l1(torch.abs(pred_fft) + eps, torch.abs(target_fft) + eps)
        loss_dict['fft'] = fft_loss.item()

        preds_features = self.vgg(preds)
        targets_features = self.vgg(targets)
        perc_loss = 0
        for i, (pf, tf) in enumerate(zip(preds_features, targets_features)):
            perc_loss += self.l1(pf, tf) * (1.0 / (2 ** i))
        loss_dict['perc'] = perc_loss.item()

        if self.use_pyiqa:
            ssim_score = self.ssim_metric(preds, targets)
            ssim_loss = 1.0 - torch.mean(ssim_score)
        else:
            ssim_loss = 1.0 - self.compute_ssim(preds, targets)
        loss_dict['ssim'] = ssim_loss.item()

        id_loss = self.id_loss_fn(preds, targets)
        loss_dict['id'] = id_loss.item()

        pred_edge = self.compute_edge(preds)
        target_edge = self.compute_edge(targets)
        edge_loss = self.l1(pred_edge, target_edge)
        loss_dict['edge'] = edge_loss.item()

        total_loss = (5.0 * pixel_loss + self.fft_weight * fft_loss +
                      self.perceptual_weight * perc_loss + self.ssim_weight * ssim_loss +
                      self.identity_weight * id_loss + self.edge_weight * edge_loss)
        return total_loss, loss_dict


# ==========================================
# 9. Dataset & Training Pipeline
# ==========================================
class UniversalRestorationDataset(Dataset):
    def __init__(self, data_config, patch_size=256, is_train=True):
        self.samples = []
        self.patch_size = patch_size
        self.is_train = is_train
        self.task_to_id = {name: i for i, name in enumerate(data_config.keys())}
        for task_name, paths in data_config.items():
            input_dir = Path(paths['input'])
            gt_dir = Path(paths['gt'])
            input_files = sorted(list(input_dir.glob('*.*')))[:2500]
            gt_files = sorted(list(gt_dir.glob('*.*')))[:2500]
            if len(input_files) > 0:
                for inp, gt in zip(input_files, gt_files):
                    self.samples.append({'input_path': str(inp), 'gt_path': str(gt), 'task_name': task_name, 'task_id': self.task_to_id[task_name]})

    def __len__(self):
        return len(self.samples)

    def transform(self, image, gt):
        w, h = image.size
        if w < self.patch_size or h < self.patch_size:
            new_w = max(w, self.patch_size)
            new_h = max(h, self.patch_size)
            image = TF.resize(image, [new_h, new_w])
            gt    = TF.resize(gt,    [new_h, new_w])
            w, h  = image.size

        if self.is_train:
            left = random.randint(0, w - self.patch_size)
            top  = random.randint(0, h - self.patch_size)
            image = image.crop((left, top, left + self.patch_size, top + self.patch_size))
            gt    = gt.crop((left, top, left + self.patch_size, top + self.patch_size))
            if random.random() > 0.5:
                image, gt = TF.hflip(image), TF.hflip(gt)
            if random.random() > 0.5:
                image, gt = TF.vflip(image), TF.vflip(gt)
        else:
            left = (w - self.patch_size) // 2
            top  = (h - self.patch_size) // 2
            image = image.crop((left, top, left + self.patch_size, top + self.patch_size))
            gt    = gt.crop((left, top, left + self.patch_size, top + self.patch_size))

        return TF.to_tensor(image), TF.to_tensor(gt)

    def __getitem__(self, idx):
        item = self.samples[idx]
        inp_img = Image.open(item['input_path']).convert('RGB')
        gt_img = Image.open(item['gt_path']).convert('RGB')
        inp_tensor, gt_tensor = self.transform(inp_img, gt_img)
        return {'input': inp_tensor, 'gt': gt_tensor, 'task_name': item['task_name'], 'task_id': item['task_id']}


def setup_pyiqa(device):
    import pyiqa
    return {
        'psnr': pyiqa.create_metric('psnr', device=device),
        'ssim': pyiqa.create_metric('ssim', device=device),
        'lpips': pyiqa.create_metric('lpips', device=device)
    }

class OMSTrainer:
    def __init__(self, config, data_config, val_data_config, best_checkpoint_path, last_checkpoint_path):
        self.config = config
        self.best_checkpoint_path = best_checkpoint_path
        self.last_checkpoint_path = last_checkpoint_path
        self.device = config['device']
        self.start_epoch = 0
        self.best_psnr = -1.0

        self._setup_train_data(data_config)
        self._setup_val_data(val_data_config)
        self._setup_model_and_optim()
        self.load_checkpoint()

    def _setup_train_data(self, data_config):
        self.train_dataset = UniversalRestorationDataset(data_config=data_config, patch_size=self.config['patch_size'], is_train=True)
        self.train_loader = DataLoader(self.train_dataset, batch_size=self.config['batch_size'], shuffle=True, num_workers=4, drop_last=True)

    def _setup_val_data(self, data_config):
        self.val_dataset = UniversalRestorationDataset(data_config=data_config, patch_size=self.config['patch_size'], is_train=False)
        self.val_loader = DataLoader(self.val_dataset, batch_size=4, shuffle=False, num_workers=4)

    def _setup_model_and_optim(self):
        self.model = KASIRNet().to(self.device)
        self.criterion = OmniRestorationLoss(
            device=self.device, fft_weight=self.config['fft_weight'],
            perceptual_weight=self.config['perceptual_weight'], identity_weight=self.config['identity_weight']
        ).to(self.device)
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=self.config['lr'])
        self.scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(self.optimizer, T_max=self.config['epochs'])
        self.metrics_calc = setup_pyiqa(self.device)

    def load_checkpoint(self):
        checkpoint_path = self.last_checkpoint_path
        if not os.path.exists(checkpoint_path):
            checkpoint_path = self.best_checkpoint_path
        if not os.path.exists(checkpoint_path):
            return
        checkpoint = torch.load(checkpoint_path, map_location=self.device, weights_only=False)
        self.model.load_state_dict(checkpoint['model_state_dict'])
        self.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        self.scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        self.start_epoch = checkpoint.get('epoch', 0)
        self.best_psnr = checkpoint.get('best_psnr', -1.0)

    def save_checkpoint(self, epoch, current_psnr, is_best):
        state = {
            'epoch': epoch,
            'model_state_dict': self.model.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'scheduler_state_dict': self.scheduler.state_dict(),
            'current_psnr': current_psnr,
            'best_psnr': self.best_psnr,
        }
        torch.save(state, self.last_checkpoint_path)
        if is_best:
            torch.save(state, self.best_checkpoint_path)

    def _train_epoch(self, epoch):
        self.model.train()
        total_loss = 0
        pbar = tqdm(self.train_loader, desc=f"Epoch {epoch}/{self.config['epochs']} (Train)")
        metrics = {}
        for batch in pbar:
            inputs = batch['input'].to(self.device)
            targets = batch['gt'].to(self.device)
            self.optimizer.zero_grad()
            restored, _ = self.model(inputs)
            loss, logs = self.criterion(restored, targets)
            loss.backward()
            nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
            self.optimizer.step()
            total_loss += loss.item()
            with torch.no_grad():
                metrics["psnr"] = self.metrics_calc["psnr"](restored, targets)
                metrics["ssim"] = self.metrics_calc["ssim"](restored, targets)
                pbar.set_postfix({'Loss': f'{loss.item():.4f}', 'PSNR': f'{metrics["psnr"].mean().item():.2f}'})
        self.scheduler.step()
        return total_loss / len(self.train_loader)

    @torch.no_grad()
    def _validate_epoch(self, epoch):
        self.model.eval()
        total_psnr = 0.0
        num_batches = 0
        vbar = tqdm(self.val_loader, desc=f"Epoch {epoch}/{self.config['epochs']} (Val)")
        for batch in vbar:
            inputs = batch['input'].to(self.device)
            targets = batch['gt'].to(self.device)
            restored, _ = self.model(inputs)
            psnr_value = self.metrics_calc['psnr'](restored, targets)
            total_psnr += psnr_value.mean().item()
            num_batches += 1
            vbar.set_postfix({'PSNR': f'{psnr_value.mean().item():.4f}'})
        return total_psnr / num_batches if num_batches > 0 else -1.0

    def run(self):
        for epoch in range(self.start_epoch + 1, self.config['epochs'] + 1):
            avg_loss = self._train_epoch(epoch)
            self.save_checkpoint(epoch, 0, False)
            current_psnr = self._validate_epoch(epoch)
            is_best = current_psnr > self.best_psnr
            if is_best:
                self.best_psnr = current_psnr
            self.save_checkpoint(epoch, current_psnr, is_best)


# ==========================================================================
# 11. KASIRNetInferencer — Full Explainability & Interpretability Suite
# Fully implemented to output explicit isolated files for every visualization method
# Fixed MatplotlibDeprecationWarning for get_cmap
# ==========================================================================

class KASIRNetInferencer:
    def __init__(self, model_path, output_root='/content/drive/MyDrive/Results/KASIRNet',
                 explain_max_size=512, enable_occlusion=False):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model = KASIRNet().to(self.device)
        self.model.eval()
        self.load_model(model_path)
        self.metrics_calc = setup_pyiqa(self.device)
        self.output_root = output_root
        self.explain_max_size = explain_max_size
        self.enable_occlusion = enable_occlusion

        sobel_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=torch.float32)
        sobel_y = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=torch.float32)
        self.sobel_x = sobel_x.view(1, 1, 3, 3).repeat(3, 1, 1, 1).to(self.device)
        self.sobel_y = sobel_y.view(1, 1, 3, 3).repeat(3, 1, 1, 1).to(self.device)

        self._acts = {}
        self._grads = {}
        self._hook_handles = []

        self.global_prompts = []
        self.global_metrics = {}

        os.makedirs(self.output_root, exist_ok=True)

    def load_model(self, model_path):
        if not os.path.exists(model_path): return
        try:
            checkpoint = torch.load(model_path, map_location=self.device, weights_only=False)
            self.model.load_state_dict(checkpoint.get('model_state_dict', checkpoint))
        except Exception as e:
            print(f"Failed to load model state: {e}")

    # ------------------------------------------------------------------
    # Hook Configuration
    # ------------------------------------------------------------------
    def _get_stage_modules(self):
        return {
            'head':      self.model.head_attn,
            'encoder1':  self.model.encoder1,
            'encoder2':  self.model.encoder2,
            'encoder3':  self.model.encoder3,
            'encoder4':  self.model.encoder4,
            'bottleneck': self.model.bottleneck,
            'decoder4':  self.model.decoder4,
            'decoder3':  self.model.decoder3,
            'decoder2':  self.model.decoder2,
            'decoder1':  self.model.decoder1,
        }

    def _get_srb_modules(self):
        return {name: mod for name, mod in self.model.named_modules() if isinstance(mod, SpectralResonanceBlock)}

    def _get_sim_modules(self):
        return {name: mod for name, mod in self.model.named_modules() if isinstance(mod, SemanticInjection)}

    def _get_sft_modules(self):
        return {name: mod for name, mod in self.model.named_modules() if isinstance(mod, SFTLayer)}

    def _register_hooks(self, capture_grads=False):
        self._remove_hooks()
        self._acts, self._grads = {}, {}

        def make_fwd_hook(key):
            def hook(module, inp, out):
                t = out[0] if isinstance(out, (tuple, list)) else out
                if torch.is_tensor(t):
                    self._acts[key] = t
                    if capture_grads and t.requires_grad:
                        def grad_hook(grad, k=key):
                            self._grads[k] = grad.detach()
                        t.register_hook(grad_hook)
            return hook

        for name, module in self._get_stage_modules().items():
            self._hook_handles.append(module.register_forward_hook(make_fwd_hook(f'stage/{name}')))
        for name, module in self._get_srb_modules().items():
            self._hook_handles.append(module.resonance_gate.register_forward_hook(make_fwd_hook(f'gate/{name}')))
        for name, module in self._get_sim_modules().items():
            self._hook_handles.append(module.spatial_attn.register_forward_hook(make_fwd_hook(f'sim/{name}')))
            self._hook_handles.append(module.channel_attn.register_forward_hook(make_fwd_hook(f'simch/{name}')))
        for name, module in self._get_sft_modules().items():
            self._hook_handles.append(module.sft_scale.register_forward_hook(make_fwd_hook(f'sft_scale/{name}')))
            self._hook_handles.append(module.sft_shift.register_forward_hook(make_fwd_hook(f'sft_shift/{name}')))

    def _remove_hooks(self):
        for h in self._hook_handles: h.remove()
        self._hook_handles = []

    @torch.no_grad()
    def restore(self, input_tensor):
        input_tensor = input_tensor.to(self.device).unsqueeze(0)
        restored, prompt_emb = self.model(input_tensor)
        return torch.clamp(restored, 0, 1).squeeze(0).cpu(), prompt_emb.squeeze(0).cpu()

    def _resize_for_explain(self, tensor):
        _, H, W = tensor.shape
        m = max(H, W)
        if m <= self.explain_max_size: return tensor
        scale = self.explain_max_size / m
        new_h, new_w = max(16, int(H * scale) // 16 * 16), max(16, int(W * scale) // 16 * 16)
        return torch.nn.functional.interpolate(tensor.unsqueeze(0), size=(new_h, new_w), mode='bilinear', align_corners=False).squeeze(0)

    # ------------------------------------------------------------------
    # Utility Savers for Individual Image Extractions
    # ------------------------------------------------------------------
    @staticmethod
    def _to_np(t): return t.detach().permute(1, 2, 0).cpu().numpy().clip(0, 1)

    @staticmethod
    def _normalize_map(m): return (m - m.min()) / (m.max() - m.min() + 1e-8)

    def _overlay_heatmap(self, image_np, heat_np, alpha=0.5, cmap='jet'):
        heat = self._normalize_map(heat_np)
        # Fixed DeprecationWarning: Using plt.get_cmap instead of cm.get_cmap
        heat_rgb = plt.get_cmap(cmap)(heat)[..., :3]
        return (1 - alpha) * image_np + alpha * heat_rgb

    def _save_single_img(self, img_np, save_path, cmap=None):
        if img_np.ndim == 2:
            plt.imsave(save_path, img_np, cmap=cmap)
        else:
            plt.imsave(save_path, np.clip(img_np, 0, 1))

    # ------------------------------------------------------------------
    # V1. Comparison
    # ------------------------------------------------------------------
    def vis_comparison(self, input_t, restored_t, gt_t, metrics, save_path, task_name):
        panels = [self._to_np(input_t), self._to_np(restored_t)]
        titles = [f"Input ({task_name})", f"Restored (PSNR: {metrics.get('psnr','N/A')})"]
        if gt_t is not None:
            panels.append(self._to_np(gt_t))
            titles.append("Ground Truth")
            err = np.abs(self._to_np(restored_t) - self._to_np(gt_t)).mean(axis=2)
            panels.append(err)
            titles.append(f"|Error|")

        fig, axes = plt.subplots(1, len(panels), figsize=(5 * len(panels), 5))
        for ax, img, title in zip(np.atleast_1d(axes), panels, titles):
            if img.ndim == 2:
                im = ax.imshow(img, cmap='inferno')
                plt.colorbar(im, ax=ax, fraction=0.046)
            else:
                ax.imshow(img)
            ax.set_title(title, fontsize=11)
            ax.axis('off')
        fig.tight_layout()
        plt.savefig(save_path, bbox_inches='tight', dpi=130); plt.close(fig)

        base_dir, stem = os.path.dirname(save_path), Path(save_path).stem
        self._save_single_img(panels[0], os.path.join(base_dir, f"{stem}_input.png"))
        self._save_single_img(panels[1], os.path.join(base_dir, f"{stem}_restored.png"))
        if gt_t is not None:
            self._save_single_img(panels[2], os.path.join(base_dir, f"{stem}_gt.png"))
            self._save_single_img(panels[3], os.path.join(base_dir, f"{stem}_errormap.png"), cmap='inferno')

    # ------------------------------------------------------------------
    # V2. Error Maps
    # ------------------------------------------------------------------
    def vis_error_maps(self, restored_t, gt_t, save_path):
        if gt_t is None: return
        r, g = self._to_np(restored_t), self._to_np(gt_t)
        err = np.abs(r - g)

        with torch.no_grad():
            rt, gt_tnsr = restored_t.unsqueeze(0).to(self.device), gt_t.unsqueeze(0).to(self.device)
            rl, gl = rt.mean(dim=1, keepdim=True), gt_tnsr.mean(dim=1, keepdim=True)
            k, C1, C2 = 11, 0.01 ** 2, 0.03 ** 2
            mu_r = torch.nn.functional.avg_pool2d(rl, k, 1, k // 2)
            mu_g = torch.nn.functional.avg_pool2d(gl, k, 1, k // 2)
            s_r = torch.nn.functional.avg_pool2d(rl * rl, k, 1, k // 2) - mu_r ** 2
            s_g = torch.nn.functional.avg_pool2d(gl * gl, k, 1, k // 2) - mu_g ** 2
            s_rg = torch.nn.functional.avg_pool2d(rl * gl, k, 1, k // 2) - mu_r * mu_g
            ssim_map = ((2 * mu_r * mu_g + C1) * (2 * s_rg + C2)) / ((mu_r ** 2 + mu_g ** 2 + C1) * (s_r + s_g + C2))
            ssim_map = ssim_map.squeeze().cpu().numpy()

        maps = [err.mean(axis=2), err[..., 0], err[..., 1], err[..., 2], 1 - ssim_map]
        titles = ["Mean |Error|", "Error R", "Error G", "Error B", "1 - SSIM map"]
        cmaps = ['inferno', 'Reds', 'Greens', 'Blues', 'magma']

        fig, axes = plt.subplots(1, 5, figsize=(25, 5))
        for ax, m, t, cmp_ in zip(axes, maps, titles, cmaps):
            im = ax.imshow(m, cmap=cmp_)
            ax.set_title(t, fontsize=10); ax.axis('off')
            plt.colorbar(im, ax=ax, fraction=0.046)
        plt.savefig(save_path, bbox_inches='tight', dpi=130); plt.close(fig)

        base_dir, stem = os.path.dirname(save_path), Path(save_path).stem
        for i, suffix in enumerate(['mean_err', 'R_err', 'G_err', 'B_err', 'inv_ssim']):
            self._save_single_img(maps[i], os.path.join(base_dir, f"{stem}_{suffix}.png"), cmap=cmaps[i])

    # ------------------------------------------------------------------
    # V3. Grad-CAM
    # ------------------------------------------------------------------
    def vis_gradcam(self, input_t, gt_t, save_path):
        small = self._resize_for_explain(input_t).to(self.device).unsqueeze(0)
        gt_small = None
        if gt_t is not None:
            gt_small = torch.nn.functional.interpolate(gt_t.unsqueeze(0).to(self.device), size=small.shape[2:], mode='bilinear', align_corners=False)

        self._register_hooks(capture_grads=True)
        self.model.zero_grad()

        with torch.enable_grad():
            restored, _ = self.model(small)
            if gt_small is not None:
                scalar = torch.nn.functional.l1_loss(restored, gt_small)
            else:
                scalar = restored.mean()
            scalar.backward()

        targets = ['stage/bottleneck', 'stage/decoder4', 'stage/decoder3', 'stage/decoder2', 'stage/decoder1']
        input_np = self._to_np(small.squeeze(0))
        cams, names = [], []

        base_dir, stem = os.path.dirname(save_path), Path(save_path).stem

        for key in targets:
            if key not in self._acts or key not in self._grads: continue
            act, grad = self._acts[key], self._grads[key]
            weights = grad.mean(dim=(2, 3), keepdim=True)
            cam = torch.relu((weights * act).sum(dim=1)).squeeze(0)
            cam = torch.nn.functional.interpolate(cam[None, None], size=small.shape[2:], mode='bilinear', align_corners=False).squeeze().detach().cpu().numpy()
            cams.append(cam)

            name = key.split('/')[-1]
            names.append(name)

            self._save_single_img(self._normalize_map(cam), os.path.join(base_dir, f"{stem}_{name}_gradcam.png"), cmap='jet')
            self._save_single_img(self._overlay_heatmap(input_np, cam), os.path.join(base_dir, f"{stem}_{name}_overlay.png"))

        self._remove_hooks()
        self.model.zero_grad()

        if not cams: return
        n = len(cams) + 1
        fig, axes = plt.subplots(2, n, figsize=(4 * n, 8))
        axes[0, 0].imshow(input_np); axes[0, 0].set_title("Input"); axes[0, 0].axis('off')
        axes[1, 0].axis('off')
        for i, (cam, name) in enumerate(zip(cams, names)):
            axes[0, i + 1].imshow(self._normalize_map(cam), cmap='jet'); axes[0, i + 1].set_title(f"Grad-CAM: {name}", fontsize=10); axes[0, i + 1].axis('off')
            axes[1, i + 1].imshow(self._overlay_heatmap(input_np, cam)); axes[1, i + 1].set_title(f"Overlay: {name}", fontsize=10); axes[1, i + 1].axis('off')
        fig.tight_layout(); plt.savefig(save_path, bbox_inches='tight', dpi=130); plt.close(fig)

    # ------------------------------------------------------------------
    # V4. Feature Maps
    # ------------------------------------------------------------------
    @torch.no_grad()
    def vis_feature_maps(self, input_t, save_path, topk=8):
        small = self._resize_for_explain(input_t).to(self.device).unsqueeze(0)
        self._register_hooks(capture_grads=False)
        _ = self.model(small)
        stage_keys = sorted([k for k in self._acts if k.startswith('stage/')])
        self._remove_hooks()

        n_stages = len(stage_keys)
        fig, axes = plt.subplots(n_stages, topk + 1, figsize=(2.2 * (topk + 1), 2.2 * n_stages))
        if n_stages == 1: axes = axes[None, :]
        input_np = self._to_np(small.squeeze(0))

        base_dir, stem = os.path.dirname(save_path), Path(save_path).stem

        for r, key in enumerate(stage_keys):
            feat = self._acts[key].squeeze(0)
            energy = feat.abs().mean(dim=(1, 2))
            idx = torch.argsort(energy, descending=True)[:topk]

            stage_name = key.split('/')[-1]
            axes[r, 0].imshow(input_np); axes[r, 0].set_ylabel(stage_name, fontsize=10, fontweight='bold'); axes[r, 0].set_xticks([]); axes[r, 0].set_yticks([])

            for c, ci in enumerate(idx):
                fmap = feat[ci].cpu().numpy()
                axes[r, c + 1].imshow(self._normalize_map(fmap), cmap='viridis')
                axes[r, c + 1].set_title(f"ch{ci.item()}", fontsize=7); axes[r, c + 1].axis('off')

                self._save_single_img(self._normalize_map(fmap), os.path.join(base_dir, f"{stem}_{stage_name}_ch{ci.item()}.png"), cmap='viridis')

        fig.tight_layout(); plt.savefig(save_path, bbox_inches='tight', dpi=110); plt.close(fig)

    # ------------------------------------------------------------------
    # V5. Spectral Analysis
    # ------------------------------------------------------------------
    def vis_spectral_analysis(self, input_t, restored_t, gt_t, save_path):
        def log_spectrum(t):
            gray = t.mean(dim=0)
            f = torch.fft.fftshift(torch.fft.fft2(gray))
            return torch.log1p(torch.abs(f)).cpu().numpy()

        def radial_profile(spec):
            h, w = spec.shape
            cy, cx = h // 2, w // 2
            y, x = np.indices((h, w))
            r = np.sqrt((x - cx) ** 2 + (y - cy) ** 2).astype(int)
            tbin = np.bincount(r.ravel(), spec.ravel())
            nr = np.bincount(r.ravel())
            return tbin / np.maximum(nr, 1)

        tensors = {'Input': input_t, 'Restored': restored_t}
        if gt_t is not None: tensors['Ground Truth'] = gt_t

        base_dir, stem = os.path.dirname(save_path), Path(save_path).stem
        profiles = {}

        n = len(tensors)
        fig = plt.figure(figsize=(5 * n + 6, 5))
        gs = fig.add_gridspec(1, n + 1, width_ratios=[1] * n + [1.4])

        for i, (name, t) in enumerate(tensors.items()):
            spec = log_spectrum(t)
            profiles[name] = radial_profile(spec)
            ax = fig.add_subplot(gs[0, i])
            im = ax.imshow(spec, cmap='magma')
            ax.set_title(f"log|FFT| — {name}", fontsize=11); ax.axis('off')
            plt.colorbar(im, ax=ax, fraction=0.046)

            self._save_single_img(self._normalize_map(spec), os.path.join(base_dir, f"{stem}_fft_{name.replace(' ', '_')}.png"), cmap='magma')

        ax_p = fig.add_subplot(gs[0, n])
        colors = {'Input': 'tab:red', 'Restored': 'tab:blue', 'Ground Truth': 'tab:green'}
        for name, prof in profiles.items():
            ax_p.plot(prof[:min(len(prof), 256)], label=name, color=colors.get(name), lw=1.5)
        ax_p.set_yscale('log'); ax_p.legend(); ax_p.grid(alpha=0.3)
        fig.tight_layout(); plt.savefig(save_path, bbox_inches='tight', dpi=130); plt.close(fig)

        fig_prof, ax_prof = plt.subplots(figsize=(6, 4))
        for name, prof in profiles.items():
            ax_prof.plot(prof[:min(len(prof), 256)], label=name, color=colors.get(name), lw=1.5)
        ax_prof.set_yscale('log'); ax_prof.legend(); ax_prof.grid(alpha=0.3)
        fig_prof.tight_layout()
        fig_prof.savefig(os.path.join(base_dir, f"{stem}_radial_profile.png"), dpi=130); plt.close(fig_prof)

    # ------------------------------------------------------------------
    # V6. Resonance Gates
    # ------------------------------------------------------------------
    @torch.no_grad()
    def vis_resonance_gates(self, input_t, save_path):
        small = self._resize_for_explain(input_t).to(self.device).unsqueeze(0)
        self._register_hooks(capture_grads=False)
        _ = self.model(small)
        gate_keys = sorted([k for k in self._acts if k.startswith('gate/')])
        gates = {k.replace('gate/', ''): self._acts[k].squeeze().cpu().numpy() for k in gate_keys}
        self._remove_hooks()
        if not gates: return

        n = len(gates)
        cols = 3
        rows = int(np.ceil(n / cols))
        fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 3.2 * rows))
        axes = np.atleast_2d(axes)

        base_dir, stem = os.path.dirname(save_path), Path(save_path).stem

        for i, (name, g) in enumerate(gates.items()):
            ax = axes[i // cols, i % cols]
            ax.bar(np.arange(len(g)), np.sort(g)[::-1], color='steelblue', width=1.0)
            ax.axhline(g.mean(), color='red', ls='--', lw=1)
            ax.set_title(name.replace('.', ' → '), fontsize=8); ax.set_ylim(0, 1)

            fig_bar, ax_bar = plt.subplots(figsize=(5,3))
            ax_bar.bar(np.arange(len(g)), np.sort(g)[::-1], color='steelblue', width=1.0)
            ax_bar.axhline(g.mean(), color='red', ls='--', lw=1)
            ax_bar.set_ylim(0, 1); ax_bar.set_title(name)
            fig_bar.tight_layout()
            fig_bar.savefig(os.path.join(base_dir, f"{stem}_gate_{name.replace('.', '_')}.png"), dpi=110); plt.close(fig_bar)

        for j in range(n, rows * cols): axes[j // cols, j % cols].axis('off')
        fig.tight_layout(); plt.savefig(save_path, bbox_inches='tight', dpi=110); plt.close(fig)

    # ------------------------------------------------------------------
    # V7. Semantic Attention
    # ------------------------------------------------------------------
    @torch.no_grad()
    def vis_semantic_attention(self, input_t, save_path):
        small = self._resize_for_explain(input_t).to(self.device).unsqueeze(0)
        self._register_hooks(capture_grads=False)
        _ = self.model(small)
        sim_keys = sorted([k for k in self._acts if k.startswith('sim/')])
        maps = {}
        for k in sim_keys:
            m = self._acts[k]
            m = torch.nn.functional.interpolate(m, size=small.shape[2:], mode='bilinear', align_corners=False)
            maps[k.replace('sim/', '')] = m.squeeze().cpu().numpy()
        self._remove_hooks()
        if not maps: return

        input_np = self._to_np(small.squeeze(0))
        n = len(maps)
        cols = 4
        rows = int(np.ceil((n + 1) / cols))
        fig, axes = plt.subplots(rows, cols, figsize=(4.5 * cols, 4.5 * rows))
        axes = np.atleast_2d(axes)
        axes[0, 0].imshow(input_np); axes[0, 0].set_title("Input"); axes[0, 0].axis('off')

        base_dir, stem = os.path.dirname(save_path), Path(save_path).stem

        for i, (name, m) in enumerate(maps.items(), start=1):
            ax = axes[i // cols, i % cols]
            overlay = self._overlay_heatmap(input_np, m)
            ax.imshow(overlay); ax.set_title(f"{name}", fontsize=8); ax.axis('off')

            self._save_single_img(overlay, os.path.join(base_dir, f"{stem}_sim_{name}.png"))

        for j in range(n + 1, rows * cols): axes[j // cols, j % cols].axis('off')
        fig.tight_layout(); plt.savefig(save_path, bbox_inches='tight', dpi=110); plt.close(fig)

    # ------------------------------------------------------------------
    # V8. Edges
    # ------------------------------------------------------------------
    @torch.no_grad()
    def vis_edges(self, input_t, restored_t, gt_t, save_path):
        def edges(t):
            t = t.unsqueeze(0).to(self.device)
            ex = torch.nn.functional.conv2d(t, self.sobel_x, padding=1, groups=3)
            ey = torch.nn.functional.conv2d(t, self.sobel_y, padding=1, groups=3)
            return torch.sqrt(ex ** 2 + ey ** 2 + 1e-6).mean(dim=1).squeeze(0).cpu().numpy()

        e_in, e_re = edges(input_t), edges(restored_t)
        panels = [e_in, e_re]
        titles = [f"Input edges", f"Restored edges"]
        if gt_t is not None:
            e_gt = edges(gt_t)
            panels += [e_gt, np.abs(e_re - e_gt)]
            titles += [f"GT edges", f"|Restored - GT| edge error"]

        fig, axes = plt.subplots(1, len(panels), figsize=(5 * len(panels), 5))
        for ax, p, t in zip(np.atleast_1d(axes), panels, titles):
            im = ax.imshow(p, cmap='gray' if 'error' not in t else 'inferno')
            ax.set_title(t, fontsize=10); ax.axis('off'); plt.colorbar(im, ax=ax, fraction=0.046)
        fig.tight_layout(); plt.savefig(save_path, bbox_inches='tight', dpi=130); plt.close(fig)

        base_dir, stem = os.path.dirname(save_path), Path(save_path).stem
        keys = ['input', 'restored', 'gt', 'error']
        for i, p in enumerate(panels):
            cmap_arg = 'inferno' if keys[i] == 'error' else 'gray'
            self._save_single_img(p, os.path.join(base_dir, f"{stem}_edge_{keys[i]}.png"), cmap=cmap_arg)

    # ------------------------------------------------------------------
    # V9. Histograms
    # ------------------------------------------------------------------
    def vis_histograms(self, input_t, restored_t, gt_t, save_path):
        tensors = {'Input': input_t, 'Restored': restored_t}
        if gt_t is not None: tensors['Ground Truth'] = gt_t
        colors3 = ['red', 'green', 'blue']

        fig, axes = plt.subplots(1, len(tensors) + 1, figsize=(5.5 * (len(tensors) + 1), 4.5))

        base_dir, stem = os.path.dirname(save_path), Path(save_path).stem

        for i, (name, t) in enumerate(tensors.items()):
            arr = self._to_np(t)

            fig_h, ax_h = plt.subplots()
            for c in range(3):
                axes[i].hist(arr[..., c].ravel(), bins=64, color=colors3[c], alpha=0.45, density=True)
                ax_h.hist(arr[..., c].ravel(), bins=64, color=colors3[c], alpha=0.45, density=True)
            axes[i].set_title(f"RGB histogram — {name}", fontsize=11); axes[i].grid(alpha=0.3)

            fig_h.tight_layout()
            fig_h.savefig(os.path.join(base_dir, f"{stem}_hist_{name.replace(' ', '_')}.png"), dpi=130)
            plt.close(fig_h)

        ls = {'Input': 'tab:red', 'Restored': 'tab:blue', 'Ground Truth': 'tab:green'}
        for name, t in tensors.items():
            lum = self._to_np(t).mean(axis=2).ravel()
            xs = np.sort(lum)
            axes[-1].plot(xs, np.linspace(0, 1, len(xs)), label=name, color=ls.get(name), lw=1.5)
        axes[-1].legend(); axes[-1].grid(alpha=0.3)
        fig.tight_layout(); plt.savefig(save_path, bbox_inches='tight', dpi=130); plt.close(fig)

    # ------------------------------------------------------------------
    # V10. Prompts
    # ------------------------------------------------------------------
    def vis_prompt(self, prompt_emb, save_path, task_name):
        p = prompt_emb.numpy()
        fig, axes = plt.subplots(1, 2, figsize=(16, 4.5))
        axes[0].bar(np.arange(len(p)), p, color='skyblue', width=1.0)
        axes[0].set_title(f"Degradation Prompt ({len(p)}D)", fontsize=11); axes[0].grid(alpha=0.3, axis='y')

        top = np.argsort(np.abs(p))[::-1][:20]
        axes[1].barh(np.arange(len(top)), p[top], color=['tab:blue' if v >= 0 else 'tab:red' for v in p[top]])
        axes[1].set_yticks(np.arange(len(top))); axes[1].set_yticklabels([f"dim {d}" for d in top], fontsize=7)
        axes[1].invert_yaxis(); axes[1].grid(alpha=0.3, axis='x')
        fig.tight_layout(); plt.savefig(save_path, bbox_inches='tight', dpi=130); plt.close(fig)

        base_dir, stem = os.path.dirname(save_path), Path(save_path).stem
        fig_b, ax_b = plt.subplots(figsize=(8,3))
        ax_b.bar(np.arange(len(p)), p, color='skyblue', width=1.0)
        fig_b.tight_layout(); fig_b.savefig(os.path.join(base_dir, f"{stem}_prompt_bar.png"), dpi=130); plt.close(fig_b)

    # ------------------------------------------------------------------
    # V11. Zoom Crops
    # ------------------------------------------------------------------
    def vis_zoom_crops(self, input_t, restored_t, gt_t, save_path, crop=96):
        if gt_t is None: return
        err = np.abs(self._to_np(restored_t) - self._to_np(gt_t)).mean(axis=2)
        H, W = err.shape
        eh = torch.nn.functional.avg_pool2d(torch.from_numpy(err)[None, None].float(), crop, stride=crop // 2)
        idx = torch.argmax(eh.flatten()).item()
        gw = eh.shape[-1]
        by, bx = idx // gw, idx % gw
        top = min(by * (crop // 2), max(0, H - crop))
        left = min(bx * (crop // 2), max(0, W - crop))

        def cut(t): return self._to_np(t)[top:top + crop, left:left + crop]

        fig, axes = plt.subplots(1, 4, figsize=(20, 5))
        for ax, img, title in zip(axes, [cut(input_t), cut(restored_t), cut(gt_t), err[top:top + crop, left:left + crop]], ["Input crop", "Restored crop", "GT crop", "Error crop"]):
            if img.ndim == 2: im = ax.imshow(img, cmap='inferno'); plt.colorbar(im, ax=ax, fraction=0.046)
            else: ax.imshow(img)
            ax.set_title(title, fontsize=11); ax.axis('off')
        fig.tight_layout(); plt.savefig(save_path, bbox_inches='tight', dpi=130); plt.close(fig)

        base_dir, stem = os.path.dirname(save_path), Path(save_path).stem
        self._save_single_img(cut(input_t), os.path.join(base_dir, f"{stem}_zoom_input.png"))
        self._save_single_img(cut(restored_t), os.path.join(base_dir, f"{stem}_zoom_restored.png"))
        self._save_single_img(cut(gt_t), os.path.join(base_dir, f"{stem}_zoom_gt.png"))
        self._save_single_img(err[top:top + crop, left:left + crop], os.path.join(base_dir, f"{stem}_zoom_error.png"), cmap='inferno')

    # ------------------------------------------------------------------
    # V12. Occlusion Maps
    # ------------------------------------------------------------------
    @torch.no_grad()
    def vis_occlusion(self, input_t, gt_t, save_path, patch=64, stride=64):
        if gt_t is None: return
        small = self._resize_for_explain(input_t).to(self.device).unsqueeze(0)
        gt_small = torch.nn.functional.interpolate(gt_t.unsqueeze(0).to(self.device), size=small.shape[2:], mode='bilinear', align_corners=False)
        base, _ = self.model(small)
        base_err = torch.nn.functional.l1_loss(base, gt_small).item()

        _, _, H, W = small.shape
        heat = np.zeros(((H - 1) // stride + 1, (W - 1) // stride + 1))
        mean_val = small.mean()
        for iy, y in enumerate(range(0, H, stride)):
            for ix, x in enumerate(range(0, W, stride)):
                occluded = small.clone()
                occluded[:, :, y:y + patch, x:x + patch] = mean_val
                out, _ = self.model(occluded)
                heat[iy, ix] = torch.nn.functional.l1_loss(out, gt_small).item() - base_err

        heat_full = torch.nn.functional.interpolate(torch.from_numpy(heat)[None, None].float(), size=(H, W), mode='bilinear', align_corners=False).squeeze().numpy()
        input_np = self._to_np(small.squeeze(0))

        fig, axes = plt.subplots(1, 3, figsize=(16, 5))
        axes[0].imshow(input_np); axes[0].set_title("Input"); axes[0].axis('off')
        im = axes[1].imshow(heat_full, cmap='jet')
        axes[1].set_title(f"Occlusion ΔL1"); axes[1].axis('off'); plt.colorbar(im, ax=axes[1], fraction=0.046)
        axes[2].imshow(self._overlay_heatmap(input_np, heat_full)); axes[2].set_title("Overlay"); axes[2].axis('off')
        fig.tight_layout(); plt.savefig(save_path, bbox_inches='tight', dpi=130); plt.close(fig)

        base_dir, stem = os.path.dirname(save_path), Path(save_path).stem
        self._save_single_img(heat_full, os.path.join(base_dir, f"{stem}_occlusion_heatmap.png"), cmap='jet')
        self._save_single_img(self._overlay_heatmap(input_np, heat_full), os.path.join(base_dir, f"{stem}_occlusion_overlay.png"))

    # ------------------------------------------------------------------
    # V13. SFT Layer Modulation Output
    # ------------------------------------------------------------------
    @torch.no_grad()
    def vis_sft_modulation(self, input_t, save_path):
        small = self._resize_for_explain(input_t).to(self.device).unsqueeze(0)
        self._register_hooks(capture_grads=False)
        _ = self.model(small)

        sft_scale_keys = sorted([k for k in self._acts if k.startswith('sft_scale/')])
        sft_shift_keys = sorted([k for k in self._acts if k.startswith('sft_shift/')])

        scales, shifts = {}, {}
        for k_sc, k_sh in zip(sft_scale_keys, sft_shift_keys):
            name = k_sc.replace('sft_scale/', '')
            scales[name] = self._acts[k_sc].mean(dim=1).squeeze().cpu().numpy()
            shifts[name] = self._acts[k_sh].mean(dim=1).squeeze().cpu().numpy()

        self._remove_hooks()
        if not scales: return

        n = len(scales)
        fig, axes = plt.subplots(2, n, figsize=(4 * n, 8))
        if n == 1: axes = axes[:, None]
        for i, name in enumerate(scales.keys()):
            sc_map, sh_map = scales[name], shifts[name]
            im1 = axes[0, i].imshow(sc_map, cmap='coolwarm'); axes[0, i].set_title(f"SFT Scale: {name}", fontsize=10); axes[0, i].axis('off')
            plt.colorbar(im1, ax=axes[0, i], fraction=0.046)
            im2 = axes[1, i].imshow(sh_map, cmap='viridis'); axes[1, i].set_title(f"SFT Shift: {name}", fontsize=10); axes[1, i].axis('off')
            plt.colorbar(im2, ax=axes[1, i], fraction=0.046)

        fig.tight_layout(); plt.savefig(save_path, bbox_inches='tight', dpi=130); plt.close(fig)

        base_dir, stem = os.path.dirname(save_path), Path(save_path).stem
        for name in scales.keys():
            self._save_single_img(scales[name], os.path.join(base_dir, f"{stem}_{name}_sft_scale.png"), cmap='coolwarm')
            self._save_single_img(shifts[name], os.path.join(base_dir, f"{stem}_{name}_sft_shift.png"), cmap='viridis')

    # ------------------------------------------------------------------
    # Global Reports
    # ------------------------------------------------------------------
    def vis_prompt_projection(self, prompts, labels, save_path, title):
        if len(prompts) < 3: return
        X = np.stack(prompts)
        method = "PCA"
        try:
            if len(prompts) >= 8:
                from sklearn.manifold import TSNE
                perp = max(2, min(30, len(prompts) - 1))
                X2 = TSNE(n_components=2, perplexity=perp, init='pca', learning_rate='auto', random_state=42).fit_transform(X)
                method = f"t-SNE (perplexity={perp})"
            else:
                raise ValueError("too few samples for t-SNE")
        except Exception:
            from sklearn.decomposition import PCA
            X2 = PCA(n_components=2, random_state=42).fit_transform(X)
            method = "PCA"

        uniq = sorted(set(labels))
        # Fixed DeprecationWarning: Using plt.get_cmap instead of cm.get_cmap
        cmap = plt.get_cmap('tab10', max(len(uniq), 1))
        fig, ax = plt.subplots(figsize=(8, 7))
        for i, u in enumerate(uniq):
            mask = np.array([l == u for l in labels])
            ax.scatter(X2[mask, 0], X2[mask, 1], s=40, alpha=0.8, color=cmap(i), label=u, edgecolors='k', linewidths=0.3)
        ax.set_title(f"{title}\n{method} projection of Degradation Prompts", fontsize=12, fontweight='bold')
        ax.legend(fontsize=8); ax.grid(alpha=0.3)
        fig.tight_layout(); plt.savefig(save_path, bbox_inches='tight', dpi=130); plt.close(fig)

    def vis_prompt_heatmap(self, prompts, names, save_path, title):
        if not prompts: return
        X = np.stack(prompts)
        fig, ax = plt.subplots(figsize=(14, max(3, 0.28 * len(prompts))))
        im = ax.imshow(X, aspect='auto', cmap='coolwarm', vmin=-np.abs(X).max(), vmax=np.abs(X).max())
        ax.set_yticks(np.arange(len(names))); ax.set_yticklabels(names, fontsize=6); ax.set_xlabel("prompt dimension")
        ax.set_title(f"{title} — Prompt heatmap (rows=images)", fontsize=12, fontweight='bold')
        plt.colorbar(im, ax=ax, fraction=0.025)
        fig.tight_layout(); plt.savefig(save_path, bbox_inches='tight', dpi=130); plt.close(fig)

    def vis_metrics_summary(self, metric_rows, save_path, title):
        if not metric_rows: return
        keys = ['psnr', 'ssim', 'lpips']
        fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
        for ax, k in zip(axes, keys):
            vals = [float(m[k]) for m in metric_rows if m.get(k, 'N/A') != 'N/A']
            if not vals: ax.axis('off'); continue
            ax.plot(vals, 'o-', ms=4, lw=1, color='steelblue')
            ax.axhline(np.mean(vals), color='red', ls='--', label=f"mean={np.mean(vals):.4f} ± {np.std(vals):.4f}")
            ax.set_title(k.upper(), fontsize=11); ax.set_xlabel("image index"); ax.legend(fontsize=8); ax.grid(alpha=0.3)
        fig.suptitle(f"{title} — Per-image metrics", fontsize=13, fontweight='bold')
        fig.tight_layout(rect=[0, 0, 1, 0.9]); plt.savefig(save_path, bbox_inches='tight', dpi=130); plt.close(fig)

    # ------------------------------------------------------------------
    # Core Image Explainer Routine
    # ------------------------------------------------------------------
    def explain_image(self, input_path, gt_path, dataset_dirs, task_name):
        stem = Path(input_path).stem
        input_img = Image.open(input_path).convert('RGB')
        input_tensor = TF.to_tensor(input_img)

        gt_tensor = None
        if gt_path and os.path.exists(gt_path):
            gt_img = Image.open(gt_path).convert('RGB')
            gt_tensor = TF.to_tensor(gt_img)
            if gt_tensor.shape[1:] != input_tensor.shape[1:]:
                gt_tensor = TF.resize(gt_tensor, [input_tensor.shape[1], input_tensor.shape[2]])
                gt_tensor = torch.clamp(gt_tensor, 0, 1)

        restored_tensor, prompt_emb = self.restore(input_tensor)

        metrics = {'psnr': 'N/A', 'ssim': 'N/A', 'lpips': 'N/A'}
        if gt_tensor is not None:
            with torch.no_grad():
                r, g = restored_tensor.unsqueeze(0).to(self.device), gt_tensor.unsqueeze(0).to(self.device)
                for k in ['psnr', 'ssim', 'lpips']:
                    metrics[k] = f"{self.metrics_calc[k](r, g).mean().item():.4f}"

        try:
            self.vis_comparison(input_tensor, restored_tensor, gt_tensor, metrics, os.path.join(dataset_dirs['01_comparison'], f"{stem}.png"), task_name)
            self.vis_error_maps(restored_tensor, gt_tensor, os.path.join(dataset_dirs['02_error_maps'], f"{stem}.png"))
            self.vis_gradcam(input_tensor, gt_tensor, os.path.join(dataset_dirs['03_gradcam'], f"{stem}.png"))
            self.vis_feature_maps(input_tensor, os.path.join(dataset_dirs['04_feature_maps'], f"{stem}.png"))
            self.vis_spectral_analysis(input_tensor, restored_tensor, gt_tensor, os.path.join(dataset_dirs['05_spectral_analysis'], f"{stem}.png"))
            self.vis_resonance_gates(input_tensor, os.path.join(dataset_dirs['06_resonance_gates'], f"{stem}.png"))
            self.vis_semantic_attention(input_tensor, os.path.join(dataset_dirs['07_semantic_attention'], f"{stem}.png"))
            self.vis_edges(input_tensor, restored_tensor, gt_tensor, os.path.join(dataset_dirs['08_edge_analysis'], f"{stem}.png"))
            self.vis_histograms(input_tensor, restored_tensor, gt_tensor, os.path.join(dataset_dirs['09_histograms'], f"{stem}.png"))
            self.vis_prompt(prompt_emb, os.path.join(dataset_dirs['10_prompt_analysis'], f"{stem}.png"), task_name)
            self.vis_zoom_crops(input_tensor, restored_tensor, gt_tensor, os.path.join(dataset_dirs['11_zoom_crops'], f"{stem}.png"))
            self.vis_sft_modulation(input_tensor, os.path.join(dataset_dirs['13_sft_modulation'], f"{stem}.png"))
            if self.enable_occlusion:
                self.vis_occlusion(input_tensor, gt_tensor, os.path.join(dataset_dirs['12_occlusion'], f"{stem}.png"))
        except Exception as e:
            print(f"  [warn] visualization failed for {stem}: {e}")
        finally:
            self._remove_hooks()
            if torch.cuda.is_available(): torch.cuda.empty_cache()

        return metrics, prompt_emb.numpy()

    def run_dataset_inference(self, dataset_name, lq_dir, gt_dir=None, max_images=32):
        dataset_root = os.path.join(self.output_root, dataset_name)
        sub_names = ['01_comparison', '02_error_maps', '03_gradcam', '04_feature_maps',
                     '05_spectral_analysis', '06_resonance_gates', '07_semantic_attention',
                     '08_edge_analysis', '09_histograms', '10_prompt_analysis',
                     '11_zoom_crops', '13_sft_modulation']
        if self.enable_occlusion: sub_names.append('12_occlusion')

        dataset_dirs = {}
        for s in sub_names:
            d = os.path.join(dataset_root, s)
            os.makedirs(d, exist_ok=True)
            dataset_dirs[s] = d

        lq_files_raw = sorted(glob.glob(os.path.join(lq_dir, "*")))
        gt_files_raw = sorted(glob.glob(os.path.join(gt_dir, "*"))) if gt_dir else []

        if dataset_name == 'snow':
            lq_files, gt_files = [], []
            for i, lq in enumerate(lq_files_raw):
                if 'rain' not in os.path.basename(lq).lower():
                    lq_files.append(lq)
                    if gt_dir and i < len(gt_files_raw):
                        gt_files.append(gt_files_raw[i])
        else:
            lq_files = lq_files_raw
            gt_files = gt_files_raw

        lq_files = lq_files[:max_images]
        gt_files = gt_files[:max_images] if gt_dir else []

        metric_rows, prompts, names = [], [], []
        for i, lq in enumerate(tqdm(lq_files, desc=f"Explaining [{dataset_name}]")):
            gt = gt_files[i] if i < len(gt_files) else None
            try:
                metrics, prompt = self.explain_image(lq, gt, dataset_dirs, dataset_name)
                metric_rows.append({'image': os.path.basename(lq), **metrics})
                prompts.append(prompt)
                names.append(Path(lq).stem)
                self.global_prompts.append((dataset_name, prompt))
            except Exception: pass

        if metric_rows:
            csv_path = os.path.join(dataset_root, 'metrics.csv')
            with open(csv_path, 'w', newline='') as f:
                writer = csv.DictWriter(f, fieldnames=['image', 'psnr', 'ssim', 'lpips'])
                writer.writeheader(); writer.writerows(metric_rows)

        self.vis_metrics_summary(metric_rows, os.path.join(dataset_root, 'metrics_summary.png'), dataset_name)
        self.vis_prompt_projection(prompts, [dataset_name] * len(prompts), os.path.join(dataset_root, 'prompt_projection.png'), dataset_name)
        self.vis_prompt_heatmap(prompts, names, os.path.join(dataset_root, 'prompt_heatmap.png'), dataset_name)

        self.global_metrics[dataset_name] = metric_rows
        return metric_rows

    def finalize_global_reports(self):
        if len(self.global_prompts) >= 3:
            labels = [d for d, _ in self.global_prompts]
            prompts = [p for _, p in self.global_prompts]
            self.vis_prompt_projection(prompts, labels, os.path.join(self.output_root, 'all_datasets_prompt_projection.png'), "All datasets")

        if self.global_metrics:
            keys = ['psnr', 'ssim', 'lpips']
            names = list(self.global_metrics.keys())
            fig, axes = plt.subplots(1, 3, figsize=(6 * 3, 5))
            for ax, k in zip(axes, keys):
                means, stds = [], []
                for n in names:
                    vals = [float(m[k]) for m in self.global_metrics[n] if m.get(k, 'N/A') != 'N/A']
                    means.append(np.mean(vals) if vals else 0)
                    stds.append(np.std(vals) if vals else 0)
                ax.bar(names, means, yerr=stds, capsize=4, color='steelblue', alpha=0.85)
                ax.set_title(k.upper(), fontsize=12); ax.tick_params(axis='x', rotation=30); ax.grid(alpha=0.3, axis='y')
                for x, m in zip(range(len(names)), means):
                    ax.text(x, m, f"{m:.3f}", ha='center', va='bottom', fontsize=8)
            fig.suptitle("Cross-Dataset Performance Comparison", fontsize=14, fontweight='bold')
            fig.tight_layout(rect=[0, 0, 1, 0.92])
            path = os.path.join(self.output_root, 'all_datasets_metrics_comparison.png')
            plt.savefig(path, bbox_inches='tight', dpi=130); plt.close(fig)
            print(f"Global comparison saved: {path}")


# ==========================================================================
# 12. Execution Pipeline
# ==========================================================================

CHECKPOINT_DIR = '/content/drive/MyDrive/Checkpoints/KASIRNet'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
BEST_CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, 'best.pth')
LAST_CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, 'last_oms_net.pth')

DATA_CONFIG = {
    'noise_sp': {
        'input': '/kaggle/input/salt-and-pepper-noise-images/Noisy_folder',
        'gt':    '/kaggle/input/salt-and-pepper-noise-images/Ground_truth',
    },
    'blur': {
        'input': '/kaggle/input/a-curated-list-of-image-deblurring-datasets/DBlur/TextOCR/train/blur',
        'gt':    '/kaggle/input/a-curated-list-of-image-deblurring-datasets/DBlur/TextOCR/train/sharp',
    },
    'rain': {
        'input': '/kaggle/input/deraindataset/train/data',
        'gt':    '/kaggle/input/deraindataset/train/gt',
    },
    'rainy': {
        'input': '/kaggle/input/jrdr-deraining-dataset/JRDR/rain_data_train_Heavy/norain',
        'gt':    '/kaggle/input/jrdr-deraining-dataset/JRDR/rain_data_train_Heavy/rain/X2',
    },
    'blurred': {
        'input': '/kaggle/input/kohlerdeblur/kohler/blur',
        'gt':    '/kaggle/input/kohlerdeblur/kohler/sharp',
    },
    'raining': {
        'input': '/kaggle/input/image-deraining/Dataset/input',
        'gt':    '/kaggle/input/image-deraining/Dataset/target',
    },
    'haze': {
        'input': '/kaggle/input/reside-6k/RESIDE-6K/train/hazy',
        'gt':    '/kaggle/input/reside-6k/RESIDE-6K/train/GT',
    },
    'gopro': {
        'input': '/kaggle/input/a-curated-list-of-image-deblurring-datasets/DBlur/Gopro/train/blur',
        'gt':    '/kaggle/input/a-curated-list-of-image-deblurring-datasets/DBlur/Gopro/train/sharp',
    },
    'hide': {
        'input': '/kaggle/input/a-curated-list-of-image-deblurring-datasets/DBlur/HIDE/train/blur',
        'gt':    '/kaggle/input/a-curated-list-of-image-deblurring-datasets/DBlur/HIDE/train/sharp',
    },
    'celeb': {
        'input': '/kaggle/input/a-curated-list-of-image-deblurring-datasets/DBlur/CelebA/train/blur',
        'gt':    '/kaggle/input/a-curated-list-of-image-deblurring-datasets/DBlur/CelebA/train/sharp',
    },
    'helen': {
        'input': '/kaggle/input/a-curated-list-of-image-deblurring-datasets/DBlur/Helen/train/blur',
        'gt':    '/kaggle/input/a-curated-list-of-image-deblurring-datasets/DBlur/Helen/train/sharp',
    },
    'snow': {
        'input': '',
        'gt':    '',
    },
    'noise': {
        'input': '/kaggle/input/image-denoising/train',
        'gt':    '/kaggle/input/image-denoising/train_cleaned',
    },
    'noisy': {
        'input': '/kaggle/input/denoising-dataset-multiple-iso-levels/AlphaISP - Denoising Dataset/AlphaISP - Denoising Dataset/PNG Data/2DNR Denoising/Noised',
        'gt':    '/kaggle/input/denoising-dataset-multiple-iso-levels/AlphaISP - Denoising Dataset/AlphaISP - Denoising Dataset/PNG Data/2DNR Denoising/Ground Truth',
    },
}

VAL_DATA_CONFIG = {
    'blur': {
        'input': '/kaggle/input/a-curated-list-of-image-deblurring-datasets/DBlur/Gopro/test/blur',
        'gt':    '/kaggle/input/a-curated-list-of-image-deblurring-datasets/DBlur/Gopro/test/sharp',
    },
    'rain': {
        'input': '/kaggle/input/deraindataset/test_b/data',
        'gt':    '/kaggle/input/deraindataset/test_b/gt',
    },
    'haze': {
        'input': '/kaggle/input/reside-6k/RESIDE-6K/test/hazy',
        'gt':    '/kaggle/input/reside-6k/RESIDE-6K/test/GT',
    },
}

TEST_DATA_CONFIG = {
    'kohler_blur': {
        'input': '',
        'gt':    '',
    },
    'gopro_blur': {
        'input': '/kaggle/input/a-curated-list-of-image-deblurring-datasets/DBlur/Gopro/test/blur',
        'gt':    '/kaggle/input/a-curated-list-of-image-deblurring-datasets/DBlur/Gopro/test/sharp',
    },
    'helen_blur': {
        'input': '/kaggle/input/a-curated-list-of-image-deblurring-datasets/DBlur/Helen/test/blur',
        'gt':    '/kaggle/input/a-curated-list-of-image-deblurring-datasets/DBlur/Helen/test/sharp',
    },
    'hide_blur': {
        'input': '/kaggle/input/a-curated-list-of-image-deblurring-datasets/DBlur/HIDE/test/blur',
        'gt':    '/kaggle/input/a-curated-list-of-image-deblurring-datasets/DBlur/HIDE/test/sharp',
    },
    'text_ocr_blur': {
        'input': '/kaggle/input/a-curated-list-of-image-deblurring-datasets/DBlur/TextOCR/test/blur',
        'gt':    '/kaggle/input/a-curated-list-of-image-deblurring-datasets/DBlur/TextOCR/test/sharp',
    },
    'celeba': {
        'input': '/kaggle/input/a-curated-list-of-image-deblurring-datasets/DBlur/CelebA/test/blur',
        'gt':    '/kaggle/input/a-curated-list-of-image-deblurring-datasets/DBlur/CelebA/test/sharp',
    },
    'rain_test_a': {
        'input': '/kaggle/input/deraindataset/test_a/data',
        'gt':    '/kaggle/input/deraindataset/test_a/gt',
    },
    'rain_test_b': {
        'input': '/kaggle/input/deraindataset/test_b/data',
        'gt':    '/kaggle/input/deraindataset/test_b/gt',
    },
    'reside_haze': {
        'input': '/kaggle/input/reside-6k/RESIDE-6K/test/hazy',
        'gt':    '/kaggle/input/reside-6k/RESIDE-6K/test/GT',
    },
    'snow': {
        'input': '',
        'gt':    '',
    },
    'noise': {
        'input': '',
        'gt':    '',
    },
    'noisy': {
        'input': '',
        'gt':    '',
    },
    'noise_sp': {
        'input': '',
        'gt':    '',
    },
}

HYPERPARAMS = {
    'batch_size': 64,
    'patch_size': 256,
    'lr': 1e-4,
    'epochs': 3000,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'fft_weight': 1.0,
    'perceptual_weight': 1.0,
    'identity_weight': 1.0,
}

if __name__ == "__main__":
    trainer = OMSTrainer(
        config=HYPERPARAMS,
        data_config=DATA_CONFIG,
        val_data_config=VAL_DATA_CONFIG,
        best_checkpoint_path=BEST_CHECKPOINT_PATH,
        last_checkpoint_path=LAST_CHECKPOINT_PATH,
    )
    trainer.run()

    saved_path = BEST_CHECKPOINT_PATH
    RESULTS_ROOT = '/content/drive/MyDrive/Results/KASIRNet'

    inferencer = KASIRNetInferencer(
        model_path=saved_path,
        output_root=RESULTS_ROOT,
        explain_max_size=512,
        enable_occlusion=True
    )

    for dataset_name, paths in TEST_DATA_CONFIG.items():
        if not os.path.isdir(paths['input']):
            print(f"[skip] {dataset_name}: {paths['input']} not found")
            continue
        inferencer.run_dataset_inference(
            dataset_name=dataset_name,
            lq_dir=paths['input'],
            gt_dir=paths.get('gt'),
            max_images=256,
        )

    inferencer.finalize_global_reports()
    print(f"\nAll explainability outputs natively isolated and saved under: {RESULTS_ROOT}")

Loading pretrained model LPIPS from /root/.cache/torch/hub/pyiqa/LPIPS_v0.1_alex-df73285e.pth


Explaining [text_ocr_blur]: 100%|██████████| 6/6 [39:46<00:00, 397.72s/it]


Global comparison saved: /content/drive/MyDrive/Results/OMSTNet/Model9/all_datasets_metrics_comparison.png

All explainability outputs natively isolated and saved under: /content/drive/MyDrive/Results/OMSTNet/Model9


In [ ]:
exit()